# ViMamba-SER | Phase 1 — Data Preparation (Colab Runner)
> Chạy trên Google Colab với GPU T4
> Repo: https://github.com/QuangHuy1911/ViMamba-SER

In [ ]:
import subprocess
result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if result.returncode == 0:
    print("\u2705 GPU available")
    print(result.stdout.split("\n")[8])  # GPU name line
else:
    raise RuntimeError("\u274c No GPU detected. Go to Runtime > Change runtime type > T4 GPU")

In [ ]:
import os
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Create output dirs on Drive (persists across disconnects)
DRIVE_DIR = "/content/drive/MyDrive/ViMamba-SER-outputs"
for folder in ["embeddings", "figures", "runs"]:
    os.makedirs(f"{DRIVE_DIR}/{folder}", exist_ok=True)
os.environ["DRIVE_DIR"] = DRIVE_DIR

print(f"\u2705 Drive mounted")
print(f"\u2705 Output dir: {DRIVE_DIR}")

# Clone repo (skip if already cloned)
if not os.path.exists("/content/ViMamba-SER"):
    !git clone https://github.com/QuangHuy1911/ViMamba-SER.git /content/ViMamba-SER
    print("\u2705 Repo cloned")
else:
    !git -C /content/ViMamba-SER pull
    print("\u2705 Repo updated")

%cd /content/ViMamba-SER

In [ ]:
!pip install -r requirements.txt -q
print("\u2705 Dependencies installed")

In [ ]:
import sys
sys.path.append("/content/ViMamba-SER/src")
from config import *

import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from collections import Counter
import torch

# Fix seed
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"\u2705 Config loaded")
print(f"   DEVICE     : {DEVICE}")
print(f"   SEED       : {SEED}")
print(f"   EMBED_DIR  : {EMBED_DIR}")
print(f"   FIGURES_DIR: {FIGURES_DIR}")

## Step 1 — Load Dataset

In [ ]:
print("Loading ViSEC from HuggingFace (first time ~2-3 mins)...")
ds = load_dataset(DATASET_NAME, split="train")

print(f"\n\u2705 Dataset loaded")
print(f"   Total samples : {len(ds)}")
print(f"   Columns       : {ds.column_names}")
print(f"\nSample[0]:")
s = ds[0]
for k, v in s.items():
    if k == "audio":
        print(f"  audio : array={v['array'].shape}, sr={v['sampling_rate']}Hz")
    else:
        print(f"  {k:12s}: {v}")

## Step 2 — Verify Labels

In [ ]:
emotions = ds["emotion"]
unique_emotions = sorted(set(emotions))
label_counts = Counter(emotions)

print(f"Unique emotions : {unique_emotions}")
print(f"\nLabel distribution:")
for emo, cnt in sorted(label_counts.items()):
    bar = "\u2588" * (cnt // 50)
    print(f"  {emo:10s}: {cnt:5d}  {bar}")

missing = [e for e in unique_emotions if e not in LABEL_MAP]
if missing:
    raise ValueError(f"\u274c Labels not in LABEL_MAP: {missing}")
else:
    print(f"\n\u2705 All labels match config LABEL_MAP")

## Step 3 — Stratified Split 70/15/15

In [ ]:
all_indices = list(range(len(ds)))
all_labels  = [LABEL_MAP[e] for e in ds["emotion"]]

train_idx, temp_idx = train_test_split(
    all_indices, test_size=0.30,
    stratify=all_labels, random_state=SEED
)
temp_labels = [all_labels[i] for i in temp_idx]
val_idx, test_idx = train_test_split(
    temp_idx, test_size=0.50,
    stratify=temp_labels, random_state=SEED
)

print(f"Split results:")
print(f"  Train : {len(train_idx):5d} ({len(train_idx)/len(ds)*100:.1f}%)")
print(f"  Val   : {len(val_idx):5d} ({len(val_idx)/len(ds)*100:.1f}%)")
print(f"  Test  : {len(test_idx):5d} ({len(test_idx)/len(ds)*100:.1f}%)")
print(f"  Total : {len(train_idx)+len(val_idx)+len(test_idx):5d}")

## Step 4 — Verify Label Distribution Per Split

In [ ]:
def count_labels(indices):
    return Counter([all_labels[i] for i in indices])

splits_counts = {
    "Train": count_labels(train_idx),
    "Val"  : count_labels(val_idx),
    "Test" : count_labels(test_idx),
}

df_dist = pd.DataFrame(splits_counts).T
df_dist.columns = LABEL_NAMES
df_dist["Total"] = df_dist.sum(axis=1)
print("Label distribution per split:")
print(df_dist.to_string())

## Step 5 — Plot Label Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
colors = ["#4CAF50", "#2196F3", "#FF9800", "#F44336"]

for ax, (name, idx) in zip(axes, [
    ("Train", train_idx), ("Val", val_idx), ("Test", test_idx)
]):
    counts = count_labels(idx)
    bars = ax.bar(LABEL_NAMES,
                  [counts[v] for v in range(NUM_CLASSES)],
                  color=colors)
    ax.set_title(f"{name} ({len(idx)} samples)", fontsize=12)
    ax.set_xlabel("Emotion")
    ax.set_ylabel("Count")
    for bar, val in zip(bars, [counts[v] for v in range(NUM_CLASSES)]):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 3, str(val),
                ha="center", fontsize=10)

plt.suptitle("ViSEC Label Distribution per Split", fontsize=14)
plt.tight_layout()

save_path = f"{FIGURES_DIR}/phase1_label_distribution.png"
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
assert os.path.exists(save_path), "\u274c Chart not saved!"
print(f"\u2705 Saved: {save_path}")

## Step 6 — Verify Audio Format

In [ ]:
import random as rnd
check_idx = rnd.sample(all_indices, 5)
sr_set = set()

print("Checking 5 random samples:\n")
for i in check_idx:
    audio = ds[i]["audio"]
    sr    = audio["sampling_rate"]
    dur   = len(audio["array"]) / sr
    sr_set.add(sr)
    print(f"  [{i:4d}] sr={sr}Hz | dur={dur:.2f}s | emotion={ds[i]['emotion']}")

if sr_set == {16000}:
    print(f"\n\u2705 All samples are 16kHz \u2014 no resampling needed")
else:
    print(f"\n\u26a0\ufe0f  Found sr: {sr_set} \u2014 resampling will be needed in Phase 2")

## Step 7 — Duration Statistics

In [ ]:
durations = np.array(ds["duration"])

print("Duration statistics (seconds):")
print(f"  Mean   : {durations.mean():.2f}s")
print(f"  Std    : {durations.std():.2f}s")
print(f"  Min    : {durations.min():.2f}s")
print(f"  Max    : {durations.max():.2f}s")
print(f"  Median : {np.median(durations):.2f}s")

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(durations, bins=40, color="#2196F3", edgecolor="white")
ax.axvline(durations.mean(), color="red", linestyle="--",
           label=f"Mean = {durations.mean():.2f}s")
ax.set_title("ViSEC Audio Duration Distribution")
ax.set_xlabel("Duration (s)")
ax.set_ylabel("Count")
ax.legend()
plt.tight_layout()

save_path = f"{FIGURES_DIR}/phase1_duration_distribution.png"
plt.savefig(save_path, dpi=150)
plt.show()
assert os.path.exists(save_path)
print(f"\u2705 Saved: {save_path}")

## Step 8 — Save splits.json (Phase 2 & 3 reuse this file)

In [ ]:
splits_data = {
    "train"    : train_idx,
    "val"      : val_idx,
    "test"     : test_idx,
    "label_map": LABEL_MAP,
    "seed"     : SEED,
    "dataset"  : DATASET_NAME,
    "stats": {
        "total": len(ds),
        "train": len(train_idx),
        "val"  : len(val_idx),
        "test" : len(test_idx),
    }
}

save_path = f"{EMBED_DIR}/splits.json"
with open(save_path, "w") as f:
    json.dump(splits_data, f, indent=2)

assert os.path.exists(save_path), "\u274c splits.json not saved!"
print(f"\u2705 Saved: {save_path}")

# Verify contents
with open(save_path) as f:
    check = json.load(f)
print(f"\nVerification:")
print(f"  train samples : {check['stats']['train']}")
print(f"  val samples   : {check['stats']['val']}")
print(f"  test samples  : {check['stats']['test']}")
print(f"  seed          : {check['seed']}")

## \u2705 Phase 1 Checklist

In [ ]:
checks = {
    "GPU available"                  : torch.cuda.is_available(),
    "Dataset loaded (>5000 samples)" : len(ds) > 5000,
    "All labels in LABEL_MAP"        : len(missing) == 0,
    "splits.json saved"              : os.path.exists(f"{EMBED_DIR}/splits.json"),
    "label_distribution.png saved"   : os.path.exists(f"{FIGURES_DIR}/phase1_label_distribution.png"),
    "duration_distribution.png saved": os.path.exists(f"{FIGURES_DIR}/phase1_duration_distribution.png"),
    "Audio is 16kHz"                 : sr_set == {16000},
    "Train ratio ~70%"               : abs(len(train_idx)/len(ds) - 0.70) < 0.02,
}

print("=" * 45)
print("       Phase 1 \u2014 Final Checklist")
print("=" * 45)
all_passed = True
for item, passed in checks.items():
    icon = "\u2705" if passed else "\u274c"
    print(f"  {icon} {item}")
    if not passed:
        all_passed = False

print("=" * 45)
if all_passed:
    print("\U0001f389 Phase 1 PASSED \u2014 G\u1edfi screenshot n\u00e0y cho Quang Huy")
    print(f"\nK\u1ebft qu\u1ea3 c\u1ea7n g\u1edfi:")
    print(f"  Total samples : {len(ds)}")
    print(f"  Train/Val/Test: {len(train_idx)}/{len(val_idx)}/{len(test_idx)}")
    print(f"  Sampling rate : {sr_set}")
    print(f"  Drive output  : {DRIVE_DIR}")
else:
    print("\u26a0\ufe0f  C\u00f3 item ch\u01b0a pass \u2014 KH\u00d4NG chuy\u1ec3n sang Phase 2")
    print("   Ch\u1ee5p m\u00e0n h\u00ecnh v\u00e0 g\u1edfi cho Quang Huy \u0111\u1ec3 ki\u1ec3m tra")